# 分类器引导 (Classifier Guidance) 的核心机制

在扩散模型中，分类器引导（Classifier Guidance）的核心思想是：**利用一个外部定义的分类器（或能量函数）的梯度，来“修正”模型原本的无条件生成路径，使其朝着我们预期的目标区域（如某个特定类别）靠拢。**

---

## 1. 符号前置：得分函数与噪声模型的关系

在阅读引导相关论文时，最先需要明确的是**得分函数（Score Function）**与**预测噪声（$\epsilon_\theta$）**的恒等关系。

### 1.1 分数（Score）的物理意义
对于加噪后的分布 $p(x_t|x_0)$，前向扩散公式为：
$$
x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon, \quad \epsilon \sim \mathcal{N}(0, \mathbf{I})
$$
这是一个标准的高斯分布，其概率密度为 $p(x_t|x_0) = \mathcal{N}(\sqrt{\bar{\alpha}_t} x_0, (1 - \bar{\alpha}_t)\mathbf{I})$。对其取对数再求梯度（即 Score）：
$$
\nabla_{x_t} \log p(x_t|x_0) = -\frac{x_t - \sqrt{\bar{\alpha}_t} x_0}{1 - \bar{\alpha}_t} = -\frac{\sqrt{1 - \bar{\alpha}_t} \epsilon}{1 - \bar{\alpha}_t} = -\frac{\epsilon}{\sqrt{1 - \bar{\alpha}_t}}
$$

### 1.2 核心等价公式
将上述公式推广到边缘分布 $p(x_t)$ 上，神经网络预测的噪声 $\epsilon_\theta(x_t, t)$ 本质上就是在拟合负的对数概率梯度：
$$
\nabla_{x_t} \log p(x_t) \approx -\frac{1}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t)
$$
> **直觉理解**：得分函数的方向是“概率密度增加最快的方向”（让图像真实性最高的演化方向），而 $\epsilon$ 的方向是“破坏图像的纯噪声方向”。因此，得分的方向恰好是噪声的相反方向，并在数值上用当前时刻的标准差进行了缩放。

---

## 2. 核心数学逻辑：基于贝叶斯定理的得分拆解

分类器引导的目标是从条件概率分布 $p(x_t|y)$ 中进行采样。得分函数（Score Function）在这里展示出了极其优美的**可加性**。

### 2.1 贝叶斯概率重塑
根据贝叶斯定理：
$$
p(x_t|y) = \frac{p(x_t)p(y|x_t)}{p(y)}
$$
两边对 $x_t$ 求对数梯度，分母 $p(y)$ 为常数项（梯度为0）：
$$
\nabla_{x_t} \log p(x_t|y) = \underbrace{\nabla_{x_t} \log p(x_t)}_{\text{无条件得分 (UNet预测)}} + \underbrace{\nabla_{x_t} \log p(y|x_t)}_{\text{分类器引导梯度}}
$$
- **无条件得分**：告诉模型如何修改像素，使得当前噪声 $x_t$ 看起来更像一张任何可能的“真实图片”。
- **分类器引导梯度**：告诉模型如何修改像素，使得该图被判定为类别 $y$ 的概率变得更高。

### 2.2 引入引导尺度 (Guidance Scale)
为了在“真实性”和“类别贴合度”之间做权衡，我们引入引导超参数 $s$：
$$
\nabla_{x_t} \log \tilde{p}(x_t|y) = \nabla_{x_t} \log p(x_t) + s \cdot \nabla_{x_t} \log p(y|x_t)
$$
- **$s=1$**：严格符合数学推导的条件概率分布。
- **$s>1$**：人为加强分类器意图。在概率地图上加深目标类别 $y$ 的“能量低谷”，迫使生成过程更大概率地掉进目标类别（这通常会提高引导准确率，但可能降低生成样本的多样性，即 truncation trick 效应）。

### 2.3 转换回噪声预测形式
将等价公式代入上述修正得分，两边同乘 $-\sqrt{1-\bar{\alpha}_t}$，即可得到在实际代码中用来代替原噪声的修正噪声公式：
$$
\boxed{\hat{\epsilon}_\theta(x_t, y, t) = \epsilon_\theta(x_t, t) - s \cdot \sqrt{1-\bar{\alpha}_t} \nabla_{x_t} \log p(y|x_t)}
$$

---

## 3. 分类器求导计算：工程实现与链式法则

虽然类别 $y$ 是离散的标签，但分类器（如 ResNet 或 ViT）在网络最后一层输出的张量（Logits）对于输入输入图像却是**连续可微**的。

### 3.1 梯度回传路径
设分类器网络为 $\Phi(x_t, t)$，其最后一层输出为 Logits 向量 $z = [z^{(1)}, z^{(2)}, \dots, z^{(K)}]$。
为了提高数值稳定性，通常使用 Log-Softmax 计算目标类别 $y = c$ 的对数概率：
$$
L = \log p(y=c|x_t) = \log \left( \frac{\exp(z^{(c)})}{\sum_{j} \exp(z^{(j)})} \right) = z^{(c)} - \log \sum_{j} \exp(z^{(j)})
$$

我们需要求该 $L$ 对于输入图像 $x_t$ 的导数，利用链式法则：
$$
g = \nabla_{x_t} L = \sum_{j} \frac{\partial L}{\partial z^{(j)}} \frac{\partial z^{(j)}}{\partial x_t} = \sum_{j} \left( \mathbb{I}_{j=c} - p(y=j|x_t) \right) \nabla_{x_t} z^{(j)}
$$

> **实操小贴士**：
> 在深度学习框架（PyTorch/JAX）中，我们无需手动推导全连接和卷积的链式反传。只需固定分类器参数，将 $x_t$ 设为需要计算导数（`requires_grad=True`），计算前向 Log-Softmax 后调用 `.backward()` 函数。最终回传得到的梯度 $g$ 即为一个与图像维度完全相同的张量，指示了每一个像素如何发生微小改变，能够最快提升模型判断它属于类别 $c$ 的概率。

---

## 4. 训练规范：如何得到一个合格的“领航员”？

常规分类器（例如在干净、清晰的 ImageNet 上训练出的传统 ResNet）是**无法直接用来**为扩散模型提供引导梯度的，因为前向采样的过程中充满了高强度噪声。分类器的训练必须遵循如下“三位一体”的要求：

### 4.1 噪声分布对齐 (Noise Distribution Alignment)
分类器看到的输入，必须与扩散模型在推理时经历的轨迹彻底一致。
- **训练数据**：输入不能是干净的图像 $x_0$，必须是依照前向扩散加噪得到的中间状态 $x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon$。
- **时间步采样**：每一次训练 Batch 都要随机采样时间步 $t \in [1, T]$，确保分类器“见识过”从近乎纯噪声到近乎清晰照片的所有模糊状态。

### 4.2 时间步感知条件化 (Time-step Conditioning)
分类器的结构必须被修改为 $\Phi(x_t, t)$ 的形式。
- **原因**：随着 $t$ 改变，图像信噪比大幅波动。在 $t$ 很大时，微弱的高频信号仅仅是整体的主导噪声；而在 $t$ 极小时，同样的信号起伏可能代表图像的真实纹理。
- **实现方式**：这常通过与扩散模型相似的正弦位置编码（Positional Encoding）抽取时间步特征，然后以加法或激活缩放（AdaGN）的形式注入到每个残差块（Residual Blocks）中。

### 4.3 防止对抗性崩溃 (Adversarial Robustness & Regularization)
分类引导本质上是在高维像素空间内顺着分类器“最满意”的方向执行**梯度上升**（Gradient Ascent）。由于神经网络的高维盲区，这极易引发对抗攻击（Adversarial Exploit）。
- 若分类器脆弱或过拟合，给出的引导梯度方向会让图像生成无意义的“高频马赛克”，而分类器却能被这堆马赛克骗过，给出 99% 属于“猫”的分类概率。
- **缓解策略**：需要施加重度正则化（例如随机裁剪 Random Crop迫使分类聚焦局部），并且扩大分类网络容量使其能捕捉宽泛语义（最好与主扩散编码器相当级别），进而输出更为真实、平滑且能在视觉上识别的面片级修改梯度。